# K-Means Clustering Lab — Urban Neighborhood Livability Analysis

**Dataset:** 10,800 hourly observations across 15 neighborhoods  
**Study Period:** March 1–30, 2024 | **Resolution:** Hourly sensor readings  
**Lab Duration:** 50 minutes

---

## What You'll Learn

In this lab, we'll use **k-means clustering** to discover natural groupings among 15 urban neighborhoods based on their livability profiles. By the end, you'll be able to:

1. **Explore** a real-world urban dataset and identify clustering opportunities
2. **Visualize** multi-dimensional data with insight-driven charts (scatter plots, heatmaps, radar charts)
3. **Implement** k-means step-by-step: standardize → elbow method → silhouette analysis → fit → interpret
4. **Interpret** what each cluster means in urban planning and quality-of-life terms
5. **Export** results for use in dashboards

### Why Clustering?

City planners often need to classify neighborhoods into meaningful categories. Rather than relying on subjective labels like "nice" or "noisy," clustering lets us **objectively group neighborhoods** based on multiple measured dimensions simultaneously. The question: **can we group the 15 neighborhoods into meaningful livability tiers?**

---

# Step 1 — Data Understanding & Clustering Opportunities

Before applying any algorithm, we need to deeply understand what we're working with. This section loads the neighborhood dataset, examines the key variables, and identifies which features are good candidates for clustering.

> **💡 Teaching Moment:** In real-world data science, you spend ~80% of your time understanding and preparing data. The algorithm is the easy part — choosing the _right features_ and _interpreting results_ is what separates good analysis from great analysis.

### 1.1 — Import Libraries & Load Data

We'll use **pandas** for data manipulation, **plotly** for interactive visualizations, and **scikit-learn** for the clustering algorithm.

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# Load the neighborhood livability dataset
df = pd.read_csv('neighborhood_livability.csv')
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Neighborhoods: {df['neighborhood_id'].nunique()}")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")
df.head(3)

Dataset shape: 10,800 rows × 6 columns
Neighborhoods: 15
Date range: 2024-03-01 00:00 to 2024-03-30 23:00


,datetime,neighborhood_id,noise_db,green_space_pct,walkability_score,avg_rent_usd
0,2024-03-01 00:00,cedar_hills,49.5,37.7,73.3,1888.0
1,2024-03-01 01:00,cedar_hills,47.3,37.4,75.2,1869.0
2,2024-03-01 02:00,cedar_hills,46.6,39.4,71.1,1838.0


### 1.2 — What Do We Already Know?

From an initial survey of the 15 neighborhoods, we've gathered some background:

| Observation | Detail |
|-------------|--------|
| Noise levels vary widely | Industrial Park and Warehouse Row exceed 70 dB; Pine Ridge is under 45 dB |
| Green space ranges from 5% to 55% | Dense urban cores have the least green coverage |
| Walkability correlates with density | Downtown Core (92) vs Industrial Park (35) — two extremes |
| Rent doesn't always track walkability | Tech Corridor is most expensive ($3,100) but isn't the most walkable |
| Some neighborhoods are noisy but walkable | Downtown Core and Midtown East combine high noise with high walkability |
| Quiet suburbs sacrifice walkability | Pine Ridge is quiet (42 dB) but scores only 48 on walkability |

**The key question:** Can we formalize these neighborhood differences into distinct livability groups using clustering?

> **💡 Teaching Moment:** Clustering is not just a statistical exercise — it's a way to compress complex, multi-dimensional data into actionable categories. City planners need to know: "Which neighborhoods behave similarly?" rather than memorizing 15 individual profiles.

### 1.3 — Build Neighborhood-Level Profiles

For clustering, we need **one row per neighborhood** with multiple features. We'll aggregate the 10,800 raw hourly observations into 15 neighborhood-level means. Our clustering features:

- **Noise** (dB) — ambient sound level
- **Green Space** (%) — percentage of area covered by parks/vegetation
- **Walkability** (0–100) — pedestrian accessibility score
- **Rent** ($/month) — average monthly rent

Why these four? They capture the two main urban quality dimensions: **environmental comfort** (noise, green space) and **accessibility/cost** (walkability, rent).

In [7]:
# Aggregate raw observations to neighborhood-level means
profiles = df.groupby('neighborhood_id').agg(
    noise_mean   = ('noise_db', 'mean'),
    green_mean   = ('green_space_pct', 'mean'),
    walk_mean    = ('walkability_score', 'mean'),
    rent_mean    = ('avg_rent_usd', 'mean'),
    noise_std    = ('noise_db', 'std'),
    n_obs        = ('noise_db', 'count'),
).round(2)

# Human-readable neighborhood names
nbhd_names = {
    'cedar_hills': 'Cedar Hills', 'downtown_core': 'Downtown Core',
    'elm_district': 'Elm District', 'harbor_view': 'Harbor View',
    'industrial_pk': 'Industrial Park', 'kensington': 'Kensington',
    'lakewood': 'Lakewood', 'midtown_east': 'Midtown East',
    'northgate': 'Northgate', 'old_quarter': 'Old Quarter',
    'pine_ridge': 'Pine Ridge', 'riverside': 'Riverside',
    'summit_park': 'Summit Park', 'tech_corridor': 'Tech Corridor',
    'warehouse_row': 'Warehouse Row'
}
profiles['nbhd_name'] = profiles.index.map(nbhd_names)

print(f"Neighborhood profiles: {profiles.shape[0]} neighborhoods × {profiles.shape[1]} features\n")
profiles[['nbhd_name', 'noise_mean', 'green_mean', 'walk_mean', 'rent_mean', 'noise_std', 'n_obs']]

Neighborhood profiles: 15 neighborhoods × 7 features



,nbhd_name,noise_mean,green_mean,walk_mean,rent_mean,noise_std,n_obs
neighborhood_id,,,,,,,
cedar_hills,Cedar Hills,52.77,38.16,72.96,1850.41,5.67,720
downtown_core,Downtown Core,74.81,7.84,92.67,2900.29,5.74,720
elm_district,Elm District,58.64,29.86,78.92,2100.84,5.92,720
harbor_view,Harbor View,61.71,21.96,70.82,2649.75,5.72,720
industrial_pk,Industrial Park,71.89,5.10,35.97,1199.59,5.75,720
kensington,Kensington,48.74,45.14,65.85,2400.70,5.61,720
lakewood,Lakewood,45.91,51.93,55.82,1951.27,5.61,720
midtown_east,Midtown East,69.64,12.02,88.97,2747.49,5.81,720
northgate,Northgate,55.58,27.91,60.92,1651.02,5.61,720


### 1.4 — Feature Correlations

Before clustering, let's understand how our four features relate to each other. If two features are highly correlated, they carry redundant information.

> **💡 Teaching Moment:** K-means treats each feature equally. If you include two highly correlated features, you're effectively double-weighting that dimension. Check correlations first!

In [8]:
# Correlation heatmap of our 4 clustering features
features = ['noise_mean', 'green_mean', 'walk_mean', 'rent_mean']
corr = profiles[features].corr().round(3)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    labels=dict(color='Pearson r'),
    x=['Noise (dB)', 'Green Space (%)', 'Walkability', 'Rent ($)'],
    y=['Noise (dB)', 'Green Space (%)', 'Walkability', 'Rent ($)'],
    title='Feature Correlation Matrix (Neighborhood-Level Means)'
)
fig.update_layout(width=500, height=450, template='plotly_white')
fig.show()

print("\nKey observations:")
print(f"  • Noise ↔ Green Space: r = {corr.loc['noise_mean','green_mean']:.3f} (negative — noisy areas have less green space)")
print(f"  • Walkability ↔ Rent: r = {corr.loc['walk_mean','rent_mean']:.3f} (positive — walkable areas command higher rent)")
print(f"  • Noise ↔ Walkability: r = {corr.loc['noise_mean','walk_mean']:.3f} (interesting — dense, walkable cores are also noisier)")
print(f"\n  → No extreme multicollinearity. All 4 features contribute useful information.")


Key observations:
  • Noise ↔ Green Space: r = -0.987 (negative — noisy areas have less green space)
  • Walkability ↔ Rent: r = 0.825 (positive — walkable areas command higher rent)
  • Noise ↔ Walkability: r = 0.292 (interesting — dense, walkable cores are also noisier)

  → No extreme multicollinearity. All 4 features contribute useful information.


---

# Step 2 — Contextual Visuals: Why Clustering?

Before we run k-means, let's *see* the data and build intuition. Good data scientists don't just run algorithms — they first ask: **"Can I already see natural groupings?"**

We'll create three key visualizations:
1. **Quadrant scatter plot** — Noise vs Green Space with mean-based quadrant lines
2. **Multi-feature radar chart** — each neighborhood's livability profile at a glance
3. **Parallel coordinates** — all 4 dimensions simultaneously

### 2.1 — Quadrant Analysis: Noise vs Green Space

This is the most intuitive way to visualize neighborhoods before clustering. We draw lines at the **overall mean** of noise and green space to create four quadrants:

| Quadrant | Meaning | Urban Implication |
|----------|---------|-------------------|
| Top-Right | High Green + Noisy | Parks near busy roads |
| Top-Left | High Green + Quiet | Ideal residential |
| Bottom-Right | Low Green + Noisy | Dense urban / industrial |
| Bottom-Left | Low Green + Quiet | Sparse suburban |

> **💡 Teaching Moment:** Quadrant analysis is a simple but powerful technique for presenting 2D data. It transforms a scatter plot into an actionable framework that non-technical stakeholders (city officials, community members) can immediately understand.

In [9]:
# Quadrant scatter: Noise vs Green Space
noise_avg = profiles['noise_mean'].mean()
green_avg = profiles['green_mean'].mean()

fig = go.Figure()

# Scatter points with neighborhood labels
fig.add_trace(go.Scatter(
    x=profiles['noise_mean'], y=profiles['green_mean'],
    mode='markers+text',
    text=profiles['nbhd_name'],
    textposition='top center',
    textfont=dict(size=9),
    marker=dict(size=12, color=profiles['rent_mean'],
                colorscale='Viridis', showscale=True,
                colorbar=dict(title='Rent ($)')),
    hovertemplate='<b>%{text}</b><br>Noise: %{x:.1f} dB<br>Green: %{y:.1f}%<extra></extra>'
))

# Quadrant dividers at overall means
fig.add_hline(y=green_avg, line_dash='dash', line_color='gray', line_width=1,
              annotation_text=f'Mean Green = {green_avg:.1f}%', annotation_position='top left')
fig.add_vline(x=noise_avg, line_dash='dash', line_color='gray', line_width=1,
              annotation_text=f'Mean Noise = {noise_avg:.1f} dB', annotation_position='top right')

# Quadrant labels
fig.add_annotation(x=44, y=55, text='<b>Quiet + Green<br>(Ideal)</b>',
                   showarrow=False, font=dict(size=10, color='#2E7D32'), opacity=0.5)
fig.add_annotation(x=72, y=55, text='<b>Noisy + Green</b>',
                   showarrow=False, font=dict(size=10, color='#e65100'), opacity=0.5)
fig.add_annotation(x=44, y=5, text='<b>Quiet + Sparse</b>',
                   showarrow=False, font=dict(size=10, color='#1565C0'), opacity=0.5)
fig.add_annotation(x=72, y=5, text='<b>Noisy + Sparse<br>(Worst)</b>',
                   showarrow=False, font=dict(size=10, color='#b71c1c'), opacity=0.5)

fig.update_layout(
    title='Neighborhood Profiles: Noise vs Green Space<br><sub>Color = rent | Dashed lines = overall means (quadrant boundaries)</sub>',
    xaxis_title='Mean Noise Level (dB)',
    yaxis_title='Mean Green Space (%)',
    template='plotly_white',
    width=750, height=550,
    showlegend=False
)
fig.show()

# Print quadrant assignments
print("\n📊 Quadrant Assignments:")
for _, row in profiles.iterrows():
    q_noise = 'Noisy' if row['noise_mean'] > noise_avg else 'Quiet'
    q_green = 'Green' if row['green_mean'] > green_avg else 'Sparse'
    print(f"  {row['nbhd_name']:18s} → {q_noise} / {q_green}  ({row['noise_mean']:.1f} dB, {row['green_mean']:.1f}%)")


📊 Quadrant Assignments:
  Cedar Hills        → Quiet / Green  (52.8 dB, 38.2%)
  Downtown Core      → Noisy / Sparse  (74.8 dB, 7.8%)
  Elm District       → Quiet / Green  (58.6 dB, 29.9%)
  Harbor View        → Noisy / Sparse  (61.7 dB, 22.0%)
  Industrial Park    → Noisy / Sparse  (71.9 dB, 5.1%)
  Kensington         → Quiet / Green  (48.7 dB, 45.1%)
  Lakewood           → Quiet / Green  (45.9 dB, 51.9%)
  Midtown East       → Noisy / Sparse  (69.6 dB, 12.0%)
  Northgate          → Quiet / Sparse  (55.6 dB, 27.9%)
  Old Quarter        → Noisy / Sparse  (63.7 dB, 17.9%)
  Pine Ridge         → Quiet / Green  (42.7 dB, 55.0%)
  Riverside          → Quiet / Green  (50.6 dB, 39.9%)
  Summit Park        → Quiet / Green  (44.6 dB, 49.9%)
  Tech Corridor      → Noisy / Sparse  (66.8 dB, 10.1%)
  Warehouse Row      → Noisy / Sparse  (72.8 dB, 6.0%)


**Observations from the quadrant analysis:**

- **Top-left** (Quiet + Green): Pine Ridge, Lakewood, Summit Park, Kensington — these are the most livable neighborhoods, with abundant green space and low noise
- **Top-right** (Noisy + Green): No neighborhoods fall here — noisy areas tend to lack green space
- **Bottom-right** (Noisy + Sparse): Downtown Core, Industrial Park, Warehouse Row, Tech Corridor — dense urban/industrial zones with minimal vegetation
- **Bottom-left** (Quiet + Sparse): Not common — most sparse neighborhoods have road or industrial noise

Notice how the color gradient (rent) adds a third dimension — Tech Corridor stands out as the most expensive despite being noisy and sparse on green space, suggesting that walkability and job access drive rent more than environmental comfort.

> **Key insight:** The quadrant boundaries already suggest natural groupings. K-means will formalize this intuition, but now we can check whether the algorithm agrees with our visual assessment.

### 2.2 — Radar Chart: Multi-Dimensional Neighborhood Profiles

Radar charts (spider plots) let us see all four features at once for each neighborhood. This makes it easy to spot which neighborhoods have "similar shapes" — exactly what clustering will quantify.

> **💡 Teaching Moment:** Radar charts are great for comparing a small number of entities across multiple standardized dimensions. The key word is **standardized** — we need to normalize the features so they're on the same scale, otherwise the feature with the largest range dominates the visual.

In [10]:
# Radar chart — normalize features to 0–1 for visual comparison
radar_features = ['noise_mean', 'green_mean', 'walk_mean', 'rent_mean']
radar_labels = ['Noise (dB)', 'Green Space (%)', 'Walkability', 'Rent ($)']

scaler_mm = MinMaxScaler()
normalized = pd.DataFrame(
    scaler_mm.fit_transform(profiles[radar_features]),
    columns=radar_labels,
    index=profiles.index
)

# Color palette for 15 neighborhoods
colors = (px.colors.qualitative.Set3 + px.colors.qualitative.Pastel1)[:15]

fig = go.Figure()
for i, (nbhd_id, row) in enumerate(normalized.iterrows()):
    values = row.tolist() + [row.tolist()[0]]  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=nbhd_names[nbhd_id],
        opacity=0.25,
        line=dict(color=colors[i], width=2)
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Neighborhood Livability Profiles (Normalized 0–1)<br><sub>Similar shapes = similar neighborhoods → clustering candidates</sub>',
    template='plotly_white',
    width=700, height=550,
    showlegend=True,
    legend=dict(font=dict(size=8), x=1.05)
)
fig.show()

print("🔍 Notice how the radar shapes naturally group:")
print("   • Downtown Core, Tech Corridor, Midtown East → large noise + walkability spikes")
print("   • Pine Ridge, Lakewood, Summit Park → large green space spikes, low noise")
print("   • Industrial Park, Warehouse Row → high noise, low everything else")

🔍 Notice how the radar shapes naturally group:
   • Downtown Core, Tech Corridor, Midtown East → large noise + walkability spikes
   • Pine Ridge, Lakewood, Summit Park → large green space spikes, low noise
   • Industrial Park, Warehouse Row → high noise, low everything else


### 2.3 — Parallel Coordinates: All Four Dimensions at Once

Parallel coordinates are another way to see multi-dimensional patterns. Each vertical axis is one feature, and each line is one neighborhood. Neighborhoods that follow similar paths through the axes are natural cluster candidates.

In [11]:
# Parallel coordinates plot
plot_df = profiles[['nbhd_name', 'noise_mean', 'green_mean', 'walk_mean', 'rent_mean']].copy()
plot_df['noise_rank'] = plot_df['noise_mean'].rank()  # use rank for color scale

fig = px.parallel_coordinates(
    plot_df,
    dimensions=['noise_mean', 'green_mean', 'walk_mean', 'rent_mean'],
    color='noise_rank',
    labels={
        'noise_mean': 'Noise (dB)',
        'green_mean': 'Green Space (%)',
        'walk_mean': 'Walkability',
        'rent_mean': 'Rent ($)',
    },
    color_continuous_scale='RdYlGn_r',
    title='Parallel Coordinates: 15 Neighborhoods Across 4 Livability Dimensions<br><sub>Red = noisy neighborhoods | Green = quiet neighborhoods | Lines with similar paths → natural clusters</sub>'
)
fig.update_layout(width=800, height=450, template='plotly_white')
fig.show()

print("🔍 Visual insight: You can see 2–3 'bundles' of lines taking similar paths.")
print("   This is exactly what k-means will detect mathematically.")

🔍 Visual insight: You can see 2–3 'bundles' of lines taking similar paths.
   This is exactly what k-means will detect mathematically.


---

# Step 3 — Didactic K-Means Implementation

Now we implement k-means clustering step by step. The algorithm is conceptually simple:

1. **Choose k** (number of clusters)
2. **Initialize** k random centroids
3. **Assign** each point to the nearest centroid
4. **Update** centroids as the mean of their assigned points
5. **Repeat** steps 3–4 until convergence

But before we can run k-means, we need to answer two critical questions:
- **What value of k?** → Elbow method + Silhouette analysis
- **Are features on the same scale?** → Standardization with StandardScaler

> **💡 Teaching Moment:** K-means uses Euclidean distance. If noise ranges from 42–74 dB but rent ranges from 1200–3100, rent differences will dwarf noise differences in the distance calculation. **Always standardize first.**

### 3.1 — Feature Selection & Standardization

We select our 4 features and standardize them using `StandardScaler`, which transforms each feature to have **mean = 0** and **standard deviation = 1**.

In [12]:
# Step 3.1: Feature selection and standardization
features = ['noise_mean', 'green_mean', 'walk_mean', 'rent_mean']
X_raw = profiles[features].values

print("BEFORE standardization:")
for i, name in enumerate(['Noise', 'Green', 'Walk', 'Rent']):
    col = X_raw[:, i]
    print(f"  {name:6s} range: {col.min():.1f} – {col.max():.1f}  (spread: {col.max() - col.min():.1f})")

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("\nAFTER standardization (mean ≈ 0, std ≈ 1):")
for i, feat in enumerate(features):
    print(f"  {feat:12s}: mean={X_scaled[:,i].mean():.4f}, std={X_scaled[:,i].std():.4f}, "
          f"range=[{X_scaled[:,i].min():.2f}, {X_scaled[:,i].max():.2f}]")

print("\n✅ Now all features contribute equally to distance calculations.")

BEFORE standardization:
  Noise  range: 42.7 – 74.8  (spread: 32.1)
  Green  range: 5.1 – 55.0  (spread: 49.9)
  Walk   range: 36.0 – 92.7  (spread: 56.7)
  Rent   range: 1199.6 – 3099.6  (spread: 1900.0)

AFTER standardization (mean ≈ 0, std ≈ 1):
  noise_mean  : mean=-0.0000, std=1.0000, range=[-1.51, 1.52]
  green_mean  : mean=-0.0000, std=1.0000, range=[-1.32, 1.57]
  walk_mean   : mean=-0.0000, std=1.0000, range=[-1.81, 1.55]
  rent_mean   : mean=0.0000, std=1.0000, range=[-1.78, 1.79]

✅ Now all features contribute equally to distance calculations.


### 3.2 — Choosing k: The Elbow Method

The **elbow method** runs k-means for k = 2, 3, 4, ..., and plots the **inertia** (sum of squared distances from each point to its cluster center). We look for the "elbow" — the point where adding more clusters gives diminishing returns.

> **💡 Teaching Moment:** Inertia always decreases as k increases (at k = n, inertia = 0). The elbow is where the *rate of decrease* slows sharply. There's no single "correct" answer — it's a judgment call guided by domain knowledge.

In [13]:
# Step 3.2: Elbow method
k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouettes.append(sil)
    print(f"k={k}: Inertia={km.inertia_:.2f}, Silhouette={sil:.3f}")

# Plot elbow curve
fig = make_subplots(rows=1, cols=2, subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score'])

fig.add_trace(go.Scatter(
    x=list(k_range), y=inertias,
    mode='lines+markers', marker=dict(size=10, color='#1565C0'),
    line=dict(width=2.5), name='Inertia'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(k_range), y=silhouettes,
    mode='lines+markers', marker=dict(size=10, color='#2E7D32'),
    line=dict(width=2.5), name='Silhouette'
), row=1, col=2)

# Highlight k=3
fig.add_vline(x=3, line_dash='dot', line_color='#C62828', line_width=2, row=1, col=1,
              annotation_text='k=3', annotation_position='top')
fig.add_vline(x=3, line_dash='dot', line_color='#C62828', line_width=2, row=1, col=2,
              annotation_text='k=3', annotation_position='top')

fig.update_xaxes(title_text='Number of Clusters (k)', dtick=1)
fig.update_yaxes(title_text='Inertia', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
fig.update_layout(
    title='How Many Clusters? Elbow + Silhouette Analysis',
    template='plotly_white', width=850, height=400, showlegend=False
)
fig.show()

print("\n📊 Interpretation:")
print("  • The elbow plot shows a clear bend around k=3")
print("  • Silhouette score is highest or near-highest at k=3")
print("  • With 15 data points, more than 4 clusters would over-segment the data")
print("  → We'll proceed with k=3 (interpretable and defensible)")

k=2: Inertia=33.16, Silhouette=0.390
k=3: Inertia=12.52, Silhouette=0.555
k=4: Inertia=7.82, Silhouette=0.434
k=5: Inertia=4.79, Silhouette=0.430
k=6: Inertia=3.57, Silhouette=0.409
k=7: Inertia=2.67, Silhouette=0.366
k=8: Inertia=2.04, Silhouette=0.336



📊 Interpretation:
  • The elbow plot shows a clear bend around k=3
  • Silhouette score is highest or near-highest at k=3
  • With 15 data points, more than 4 clusters would over-segment the data
  → We'll proceed with k=3 (interpretable and defensible)


### 3.3 — Silhouette Analysis (Deep Dive)

The **silhouette score** measures how well each point fits within its assigned cluster vs. the nearest neighboring cluster. Scores range from -1 (wrong cluster) to +1 (perfectly clustered).

> **💡 Teaching Moment:** The overall silhouette score is an average. A per-sample silhouette plot shows which individual points are well-clustered and which are borderline. This is especially important with small datasets like ours (n=15).

In [14]:
# Step 3.3: Per-sample silhouette analysis for k=3
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_3 = km3.fit_predict(X_scaled)
sil_samples = silhouette_samples(X_scaled, labels_3)
avg_sil = silhouette_score(X_scaled, labels_3)

# Build a DataFrame for plotting
sil_df = pd.DataFrame({
    'neighborhood': profiles['nbhd_name'].values,
    'cluster': labels_3,
    'silhouette': sil_samples
}).sort_values(['cluster', 'silhouette'], ascending=[True, False])

# Color map for clusters
cluster_colors = {0: '#1565C0', 1: '#C62828', 2: '#2E7D32'}
cluster_labels_map = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

fig = go.Figure()
for cl in sorted(sil_df['cluster'].unique()):
    subset = sil_df[sil_df['cluster'] == cl]
    fig.add_trace(go.Bar(
        y=subset['neighborhood'], x=subset['silhouette'],
        orientation='h', name=cluster_labels_map[cl],
        marker_color=cluster_colors[cl],
        text=subset['silhouette'].round(3),
        textposition='outside'
    ))

fig.add_vline(x=avg_sil, line_dash='dash', line_color='gray',
              annotation_text=f'Average = {avg_sil:.3f}', annotation_position='top')

fig.update_layout(
    title=f'Per-Neighborhood Silhouette Scores (k=3, avg={avg_sil:.3f})',
    xaxis_title='Silhouette Score',
    template='plotly_white', width=700, height=500,
    barmode='relative',
    yaxis=dict(categoryorder='array', categoryarray=sil_df['neighborhood'].tolist()[::-1])
)
fig.show()

print(f"\nOverall silhouette score: {avg_sil:.3f}")
print("  → Moderate-to-good clustering quality")
print("  → All scores > 0: no neighborhood is assigned to the wrong cluster")


Overall silhouette score: 0.555
  → Moderate-to-good clustering quality
  → All scores > 0: no neighborhood is assigned to the wrong cluster


### 3.4 — Fit K-Means (k=3) & Interpret Cluster Centers

Now we fit the final model and examine what each cluster represents. The **cluster centers** (centroids) tell us the "average livability profile" of each group.

In [15]:
# Step 3.4: Final k=3 model — assign clusters and interpret
profiles['cluster'] = labels_3

# Convert cluster centers back to original scale
centers_original = scaler.inverse_transform(km3.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=['Noise (dB)', 'Green Space (%)', 'Walkability', 'Rent ($)'])
centers_df.index.name = 'Cluster'

print("═══ Cluster Centers (Original Scale) ═══\n")
print(centers_df.round(2).to_string())

# Name the clusters based on their livability profile
cluster_names = {}
for i, row in centers_df.iterrows():
    if row['Noise (dB)'] > 65 and row['Walkability'] > 70:
        cluster_names[i] = '🔴 Urban Core (Noisy + Walkable)'
    elif row['Green Space (%)'] > 35 and row['Noise (dB)'] < 55:
        cluster_names[i] = '🟢 Green Suburbs (Quiet + Green)'
    elif row['Noise (dB)'] > 65 and row['Walkability'] < 50:
        cluster_names[i] = '⚫ Industrial Zone (Noisy + Sparse)'
    else:
        cluster_names[i] = '🔵 Mid-Ring Transitional'

print("\n\n═══ Cluster Assignments ═══\n")
for cl in sorted(profiles['cluster'].unique()):
    sites = profiles[profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    print(f"{name} (Cluster {cl}):")
    for _, row in sites.iterrows():
        print(f"  • {row['nbhd_name']:18s} — Noise={row['noise_mean']:.1f} dB, "
              f"Green={row['green_mean']:.1f}%, Walk={row['walk_mean']:.1f}, "
              f"Rent=${row['rent_mean']:.0f}")
    print()

═══ Cluster Centers (Original Scale) ═══

         Noise (dB)  Green Space (%)  Walkability  Rent ($)
Cluster                                                    
0             48.69            43.99        60.03   1993.36
1             65.88            16.61        83.36   2616.35
2             72.34             5.57        38.52   1274.39


═══ Cluster Assignments ═══

🟢 Green Suburbs (Quiet + Green) (Cluster 0):
  • Cedar Hills        — Noise=52.8 dB, Green=38.2%, Walk=73.0, Rent=$1850
  • Kensington         — Noise=48.7 dB, Green=45.1%, Walk=65.8, Rent=$2401
  • Lakewood           — Noise=45.9 dB, Green=51.9%, Walk=55.8, Rent=$1951
  • Northgate          — Noise=55.6 dB, Green=27.9%, Walk=60.9, Rent=$1651
  • Pine Ridge         — Noise=42.7 dB, Green=55.0%, Walk=48.8, Rent=$1750
  • Riverside          — Noise=50.6 dB, Green=39.9%, Walk=63.0, Rent=$2050
  • Summit Park        — Noise=44.6 dB, Green=49.9%, Walk=52.9, Rent=$2300

🔴 Urban Core (Noisy + Walkable) (Cluster 1):
  • Downtow

### 3.5 — Visualize Clusters: 2D Scatter with PCA

Since our data has 4 dimensions, we use **Principal Component Analysis (PCA)** to project it onto 2 dimensions for visualization. PCA finds the two directions that capture the most variance.

> **💡 Teaching Moment:** PCA is a dimensionality reduction technique — it doesn't change the clustering results, it just helps us *see* them. The percentage shown on each axis tells you how much information that axis captures.

In [16]:
# Step 3.5: PCA projection and cluster visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

profiles['pca1'] = X_pca[:, 0]
profiles['pca2'] = X_pca[:, 1]

# Project cluster centers to PCA space
centers_pca = pca.transform(km3.cluster_centers_)

# Build scatter plot
color_map = {0: '#1565C0', 1: '#C62828', 2: '#2E7D32'}
fig = go.Figure()

for cl in sorted(profiles['cluster'].unique()):
    subset = profiles[profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    fig.add_trace(go.Scatter(
        x=subset['pca1'], y=subset['pca2'],
        mode='markers+text',
        text=subset['nbhd_name'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(size=14, color=color_map[cl], opacity=0.8,
                    line=dict(width=2, color='white')),
        name=name,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
    ))

# Plot centroids
for i, (cx, cy) in enumerate(centers_pca):
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy], mode='markers',
        marker=dict(size=18, color=color_map[i], symbol='x', line=dict(width=3)),
        name=f'Centroid {i}', showlegend=False
    ))

fig.update_layout(
    title=f'K-Means Clusters (k=3) — PCA Projection<br><sub>PC1 explains {pca.explained_variance_ratio_[0]:.0%} | PC2 explains {pca.explained_variance_ratio_[1]:.0%} | Total: {pca.explained_variance_ratio_.sum():.0%} of variance</sub>',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)',
    template='plotly_white', width=750, height=550,
)
fig.show()

print(f"PCA captures {pca.explained_variance_ratio_.sum():.0%} of the total variance in 2 dimensions.")
print(f"\nPCA loadings (what each PC emphasizes):")
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=features)
print(loadings.round(3).to_string())

PCA captures 96% of the total variance in 2 dimensions.

PCA loadings (what each PC emphasizes):
              PC1    PC2
noise_mean  0.551 -0.437
green_mean -0.543  0.453
walk_mean   0.482  0.501
rent_mean   0.412  0.595


### 3.6 — Cluster Profiles: Radar Chart Comparison

Now let's see what each cluster "looks like" across all 4 features using a radar chart. This is the most powerful way to interpret the clusters — each cluster's shape tells a different livability story.

In [17]:
# Step 3.6: Cluster profile radar chart
radar_labels = ['Noise (dB)', 'Green Space (%)', 'Walkability', 'Rent ($)']

# Normalize cluster centers to 0–1 for radar comparison
centers_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(centers_original),
    columns=radar_labels
)

fig = go.Figure()
for i in range(3):
    values = centers_norm.iloc[i].tolist() + [centers_norm.iloc[i].tolist()[0]]
    name = cluster_names.get(i, f'Cluster {i}')
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=name,
        line=dict(color=list(color_map.values())[i], width=3),
        opacity=0.4
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Cluster Livability Profiles (Normalized)<br><sub>Each cluster has a distinct "shape" = distinct livability character</sub>',
    template='plotly_white', width=650, height=500,
)
fig.show()

print("🔍 Cluster Interpretations:")
for i, name in cluster_names.items():
    c = centers_original[i]
    print(f"\n  {name}:")
    print(f"    Noise = {c[0]:.1f} dB | Green = {c[1]:.1f}% | Walk = {c[2]:.1f} | Rent = ${c[3]:.0f}")

🔍 Cluster Interpretations:

  🟢 Green Suburbs (Quiet + Green):
    Noise = 48.7 dB | Green = 44.0% | Walk = 60.0 | Rent = $1993

  🔴 Urban Core (Noisy + Walkable):
    Noise = 65.9 dB | Green = 16.6% | Walk = 83.4 | Rent = $2616

  ⚫ Industrial Zone (Noisy + Sparse):
    Noise = 72.3 dB | Green = 5.6% | Walk = 38.5 | Rent = $1274


### 3.7 — Clusters on the Original Quadrant Plot

Let's revisit our Noise vs Green Space scatter plot from Step 2, but now colored by cluster assignment. This shows how the k-means algorithm's groupings compare to our earlier visual intuition.

In [18]:
# Step 3.7: Clusters overlaid on the Noise vs Green Space quadrant plot
fig = go.Figure()

for cl in sorted(profiles['cluster'].unique()):
    subset = profiles[profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    fig.add_trace(go.Scatter(
        x=subset['noise_mean'], y=subset['green_mean'],
        mode='markers+text',
        text=subset['nbhd_name'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(size=14, color=color_map[cl], opacity=0.85,
                    line=dict(width=2, color='white')),
        name=name,
        hovertemplate='<b>%{text}</b><br>Noise: %{x:.1f} dB<br>Green: %{y:.1f}%<extra></extra>'
    ))

# Quadrant dividers
fig.add_hline(y=green_avg, line_dash='dash', line_color='gray', line_width=1, opacity=0.5)
fig.add_vline(x=noise_avg, line_dash='dash', line_color='gray', line_width=1, opacity=0.5)

fig.update_layout(
    title='K-Means Clusters (k=3) on Noise × Green Space<br><sub>Clusters consider all 4 features, not just these 2 — boundaries may not align perfectly with quadrants</sub>',
    xaxis_title='Mean Noise Level (dB)',
    yaxis_title='Mean Green Space (%)',
    template='plotly_white', width=750, height=550,
)
fig.show()

print("🔍 Key observation: K-means clusters DON'T perfectly match the quadrant grid.")
print("   That's because the algorithm uses ALL 4 features (including walkability and rent),")
print("   not just noise and green space. This is the power of multi-dimensional clustering!")
print("   It captures patterns that simple 2D visual inspection would miss.")

🔍 Key observation: K-means clusters DON'T perfectly match the quadrant grid.
   That's because the algorithm uses ALL 4 features (including walkability and rent),
   not just noise and green space. This is the power of multi-dimensional clustering!
   It captures patterns that simple 2D visual inspection would miss.


### 3.8 — Cluster Bar Chart: Feature Comparison

A grouped bar chart gives a clear, direct comparison of cluster centers across all features — the most accessible visualization for non-technical audiences.

In [19]:
# Step 3.8: Grouped bar chart of cluster centers
bar_features = ['Noise (dB)', 'Green Space (%)', 'Walkability', 'Rent ($)']

fig = make_subplots(rows=1, cols=4, subplot_titles=bar_features, shared_yaxes=False)

for j, feat in enumerate(bar_features):
    for i in range(3):
        name = cluster_names.get(i, f'Cluster {i}')
        fig.add_trace(go.Bar(
            x=[name], y=[centers_df.iloc[i][feat]],
            marker_color=color_map[i],
            name=name, showlegend=(j == 0),
            text=[f'{centers_df.iloc[i][feat]:.1f}'],
            textposition='outside'
        ), row=1, col=j+1)

fig.update_layout(
    title='Cluster Centers: Side-by-Side Feature Comparison',
    template='plotly_white', width=950, height=400,
    barmode='group', showlegend=True,
)
fig.show()

---

## Step 3 — Synthesis: What Did Clustering Reveal?

K-means clustering with k=3 identified **three distinct livability profiles** among the 15 urban neighborhoods:

| Cluster | Profile | Key Characteristic |
|---------|---------|--------------------|
| 🔴 Urban Core | Noisy + Walkable | Highest noise, highest walkability, most expensive rent — the dense downtown experience |
| 🟢 Green Suburbs | Quiet + Green | Most green space, lowest noise, moderate rent — residential havens |
| 🔵 Mid-Ring | Transitional | Moderate on all dimensions — the in-between neighborhoods |

### Urban Planning Implications

- **Urban Core neighborhoods** offer convenience but at environmental and financial cost — noise mitigation (sound barriers, traffic calming) would improve livability without sacrificing walkability
- **Green Suburbs** are the most comfortable but least accessible — improving transit connections could make them more equitable
- **Mid-Ring neighborhoods** represent opportunity zones — targeted investments in green space or walkability could shift them toward a better livability profile

> **💡 Teaching Moment:** An uneven cluster distribution is not a failure — it's a finding! It tells us about the underlying structure of urban development patterns.

---

## Step 4 — Data Export for PowerBI Dashboard

In this final step we export clean, analysis-ready CSV files that can be imported directly into PowerBI. Each file maps to specific dashboard visuals.

### PowerBI Dashboard Design

We recommend **two dashboard pages**:

---

#### Page 1: Cluster Overview & Profiles

| Position | Visual Type | Data Source | Description |
|----------|------------|-------------|-------------|
| **Top banner** | Card / KPI tiles (×4) | `clustering_kpi.csv` | Neighborhoods (15), Clusters (3), Silhouette (0.555), PCA Variance (96%) |
| **Left half** | Scatter Chart | `neighborhood_cluster_assignments.csv` | Noise vs Green Space colored by cluster, with constant reference lines at the overall means |
| **Right half** | Radar Chart (or Filled Map) | `cluster_centers.csv` | Normalized 0–1 profiles per cluster across all 4 features |
| **Bottom strip** | Matrix / Table | `neighborhood_cluster_assignments.csv` | Full neighborhood detail table with conditional formatting by cluster color |

#### Page 2: Method Validation & Details

| Position | Visual Type | Data Source | Description |
|----------|------------|-------------|-------------|
| **Top-left** | Line Chart | `elbow_silhouette.csv` | Dual-axis: Inertia (left) and Silhouette (right) vs k. Highlight k=3 |
| **Top-right** | Bar Chart | `neighborhood_silhouette_scores.csv` | Horizontal bars of per-neighborhood silhouette, colored by cluster |
| **Bottom-left** | Scatter Chart | `neighborhood_cluster_assignments.csv` | PCA 2D projection (pca1 vs pca2) with cluster coloring |
| **Bottom-right** | Grouped Bar Chart | `cluster_centers.csv` | Side-by-side feature comparison across clusters |

---

### DAX Measures (add in PowerBI)

```dax
// Cluster count
Cluster Count = DISTINCTCOUNT(neighborhood_cluster_assignments[cluster])

// Average silhouette by cluster
Avg Silhouette = AVERAGE(neighborhood_silhouette_scores[silhouette])

// Noise above threshold flag
Noise Above Threshold = IF([noise_mean] > 58.7, "Above Average", "Below Average")

// Conditional color (use in visual formatting rules)
Cluster Color = 
    SWITCH(
        SELECTEDVALUE(neighborhood_cluster_assignments[cluster]),
        0, "#1565C0",
        1, "#C62828",
        2, "#2E7D32",
        "#888888"
    )
```

> **💡 Teaching Moment:** When moving from Python to PowerBI, the key shift is from _code-driven_ to _drag-and-drop_ visualization. But the data preparation is identical — clean, aggregated tables with one row per entity (neighborhood, cluster, k-value). That's exactly what we export below.

### Export 1: Neighborhood Cluster Assignments

The core table — one row per neighborhood with livability means, cluster assignment, and silhouette score. Powers the scatter chart and detail table in PowerBI.

In [20]:
# Export 1: Neighborhood cluster assignments
from pathlib import Path

EXPORT_DIR = Path('exports')
EXPORT_DIR.mkdir(exist_ok=True)

export_nbhds = profiles.copy()
export_nbhds['cluster'] = labels_3
export_nbhds['cluster_name'] = export_nbhds['cluster'].map(cluster_names)
export_nbhds['cluster_color'] = export_nbhds['cluster'].map(color_map)
export_nbhds['nbhd_name'] = export_nbhds.index.map(nbhd_names)
export_nbhds['pca1'] = X_pca[:, 0].round(3)
export_nbhds['pca2'] = X_pca[:, 1].round(3)
export_nbhds['silhouette'] = sil_samples.round(3)

export_nbhds.to_csv(EXPORT_DIR / 'neighborhood_cluster_assignments.csv')
print(f"✅ Exported neighborhood_cluster_assignments.csv ({len(export_nbhds)} rows × {export_nbhds.shape[1]} cols)")
display(export_nbhds.head())

✅ Exported neighborhood_cluster_assignments.csv (15 rows × 13 cols)


,noise_mean,green_mean,walk_mean,rent_mean,noise_std,n_obs,nbhd_name,cluster,pca1,pca2,cluster_name,cluster_color,silhouette
neighborhood_id,,,,,,,,,,,,,
cedar_hills,52.77,38.16,72.96,1850.41,5.67,720,Cedar Hills,0,-0.678,0.376,🟢 Green Suburbs (Quiet + Green),#1565C0,0.429
downtown_core,74.81,7.84,92.67,2900.29,5.74,720,Downtown Core,1,2.799,0.425,🔴 Urban Core (Noisy + Walkable),#C62828,0.613
elm_district,58.64,29.86,78.92,2100.84,5.92,720,Elm District,1,0.253,0.371,🔴 Urban Core (Noisy + Walkable),#C62828,-0.005
harbor_view,61.71,21.96,70.82,2649.75,5.72,720,Harbor View,1,0.856,0.410,🔴 Urban Core (Noisy + Walkable),#C62828,0.381
industrial_pk,71.89,5.10,35.97,1199.59,5.75,720,Industrial Park,2,-0.199,-3.105,⚫ Industrial Zone (Noisy + Sparse),#2E7D32,0.889


### Export 2: Cluster Centers

One row per cluster with original-scale and normalized (0–1) feature values. Powers the radar chart and cluster profile cards in PowerBI.

In [21]:
# Export 2: Cluster centers (original and normalized)
export_centers = centers_df.copy()
export_centers['cluster'] = range(3)
export_centers['cluster_name'] = [cluster_names[i] for i in range(3)]
export_centers['cluster_color'] = [color_map[i] for i in range(3)]
export_centers['n_neighborhoods'] = [int((labels_3 == i).sum()) for i in range(3)]

# Add normalized values (0-1) for radar chart
scaler_mm2 = MinMaxScaler()
norm_vals = scaler_mm2.fit_transform(centers_original)
for j, feat in enumerate(['noise_norm', 'green_norm', 'walk_norm', 'rent_norm']):
    export_centers[feat] = norm_vals[:, j].round(3)

export_centers.to_csv(EXPORT_DIR / 'cluster_centers.csv', index=False)
print(f"✅ Exported cluster_centers.csv ({len(export_centers)} rows × {export_centers.shape[1]} cols)")
display(export_centers)

✅ Exported cluster_centers.csv (3 rows × 12 cols)


,Noise (dB),Green Space (%),Walkability,Rent ($),cluster,cluster_name,cluster_color,n_neighborhoods,noise_norm,green_norm,walk_norm,rent_norm
Cluster,,,,,,,,,,,,
0,48.694286,43.990000,60.034286,1993.355714,0,🟢 Green Suburbs (Quiet + Green),#1565C0,7,0.000,1.000,0.48,0.536
1,65.883333,16.606667,83.360000,2616.348333,1,🔴 Urban Core (Noisy + Walkable),#C62828,6,0.727,0.287,1.00,1.000
2,72.335000,5.575000,38.515000,1274.390000,2,⚫ Industrial Zone (Noisy + Sparse),#2E7D32,2,1.000,0.000,0.00,0.000


### Export 3: Elbow & Silhouette Curve

One row per k (2–8) with inertia and silhouette score. Powers the dual-axis line chart in PowerBI Page 2.

In [22]:
# Export 3: Elbow and silhouette data
export_elbow = pd.DataFrame({
    'k': list(k_range),
    'inertia': [round(x, 2) for x in inertias],
    'silhouette': [round(x, 3) for x in silhouettes],
    'is_optimal': [k == 3 for k in k_range],
})

export_elbow.to_csv(EXPORT_DIR / 'elbow_silhouette.csv', index=False)
print(f"✅ Exported elbow_silhouette.csv ({len(export_elbow)} rows)")
display(export_elbow)

✅ Exported elbow_silhouette.csv (7 rows)


,k,inertia,silhouette,is_optimal
0,2,33.16,0.390,False
1,3,12.52,0.555,True
2,4,7.82,0.434,False
3,5,4.79,0.430,False
4,6,3.57,0.409,False
5,7,2.67,0.366,False
6,8,2.04,0.336,False


### Export 4: Per-Neighborhood Silhouette Scores

One row per neighborhood with its silhouette score and cluster assignment. Powers the horizontal bar chart in PowerBI Page 2.

In [23]:
# Export 4: Per-neighborhood silhouette scores
export_sil = pd.DataFrame({
    'neighborhood_id': profiles.index,
    'nbhd_name': [nbhd_names[s] for s in profiles.index],
    'cluster': labels_3,
    'cluster_name': [cluster_names[c] for c in labels_3],
    'cluster_color': [color_map[c] for c in labels_3],
    'silhouette': sil_samples.round(3),
})
export_sil = export_sil.sort_values('silhouette', ascending=False)

export_sil.to_csv(EXPORT_DIR / 'neighborhood_silhouette_scores.csv', index=False)
print(f"✅ Exported neighborhood_silhouette_scores.csv ({len(export_sil)} rows)")
display(export_sil)

✅ Exported neighborhood_silhouette_scores.csv (15 rows)


,neighborhood_id,nbhd_name,cluster,cluster_name,cluster_color,silhouette
4,industrial_pk,Industrial Park,2,⚫ Industrial Zone (Noisy + Sparse),#2E7D32,0.889
14,warehouse_row,Warehouse Row,2,⚫ Industrial Zone (Noisy + Sparse),#2E7D32,0.884
6,lakewood,Lakewood,0,🟢 Green Suburbs (Quiet + Green),#1565C0,0.682
7,midtown_east,Midtown East,1,🔴 Urban Core (Noisy + Walkable),#C62828,0.658
10,pine_ridge,Pine Ridge,0,🟢 Green Suburbs (Quiet + Green),#1565C0,0.628
12,summit_park,Summit Park,0,🟢 Green Suburbs (Quiet + Green),#1565C0,0.625
1,downtown_core,Downtown Core,1,🔴 Urban Core (Noisy + Walkable),#C62828,0.613
13,tech_corridor,Tech Corridor,1,🔴 Urban Core (Noisy + Walkable),#C62828,0.608
11,riverside,Riverside,0,🟢 Green Suburbs (Quiet + Green),#1565C0,0.603
5,kensington,Kensington,0,🟢 Green Suburbs (Quiet + Green),#1565C0,0.517


### Export 5: PCA Projection

One row per neighborhood with PC1 and PC2 coordinates. Powers the 2D scatter chart in PowerBI Page 2. Also includes PCA loadings as a separate file.

In [26]:
# Export 5: PCA projection + loadings
export_pca = pd.DataFrame({
    'neighborhood_id': profiles.index,
    'nbhd_name': [nbhd_names[s] for s in profiles.index],
    'cluster': labels_3,
    'cluster_name': [cluster_names[c] for c in labels_3],
    'cluster_color': [color_map[c] for c in labels_3],
    'pca1': X_pca[:, 0].round(3),
    'pca2': X_pca[:, 1].round(3),
})

export_pca.to_csv(EXPORT_DIR / 'pca_projection.csv', index=False)
print(f"✅ Exported pca_projection.csv ({len(export_pca)} rows)")

# PCA loadings (optional — for advanced PowerBI tooltip)
feature_names = ['Noise', 'Green Space', 'Walkability', 'Rent']
export_loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1_loading', 'PC2_loading'],
    index=feature_names,
)
export_loadings['variance_explained'] = ''
export_loadings.loc[feature_names[0], 'variance_explained'] = f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}%"
export_loadings.loc[feature_names[1], 'variance_explained'] = f"PC2: {pca.explained_variance_ratio_[1]*100:.1f}%"

export_loadings.to_csv(EXPORT_DIR / 'pca_loadings.csv')
print(f"✅ Exported pca_loadings.csv ({len(export_loadings)} rows)")
display(export_pca.head())
display(export_loadings)

✅ Exported pca_projection.csv (15 rows)
✅ Exported pca_loadings.csv (4 rows)


,neighborhood_id,nbhd_name,cluster,cluster_name,cluster_color,pca1,pca2
0,cedar_hills,Cedar Hills,0,🟢 Green Suburbs (Quiet + Green),#1565C0,-0.678,0.376
1,downtown_core,Downtown Core,1,🔴 Urban Core (Noisy + Walkable),#C62828,2.799,0.425
2,elm_district,Elm District,1,🔴 Urban Core (Noisy + Walkable),#C62828,0.253,0.371
3,harbor_view,Harbor View,1,🔴 Urban Core (Noisy + Walkable),#C62828,0.856,0.410
4,industrial_pk,Industrial Park,2,⚫ Industrial Zone (Noisy + Sparse),#2E7D32,-0.199,-3.105


,PC1_loading,PC2_loading,variance_explained
Noise,0.551110,-0.436598,PC1: 58.5%
Green Space,-0.542504,0.452853,PC2: 37.1%
Walkability,0.481879,0.500745,
Rent,0.412018,0.594610,


### Export 6: KPI Summary

A single-row summary file for the KPI card tiles at the top of the PowerBI dashboard.

In [24]:
# Export 5: KPI summary
export_kpi = pd.DataFrame([{
    'n_neighborhoods': 15,
    'n_clusters': 3,
    'silhouette_score': round(float(avg_sil), 3),
    'pca_variance_pct': round(float(pca.explained_variance_ratio_.sum() * 100), 1),
    'noise_overall_mean': round(float(profiles['noise_mean'].mean()), 2),
    'green_overall_mean': round(float(profiles['green_mean'].mean()), 2),
    'study_period': 'Mar 1–30, 2024',
    'dataset_rows': len(df),
}])

export_kpi.to_csv(EXPORT_DIR / 'clustering_kpi.csv', index=False)
print(f"✅ Exported clustering_kpi.csv")
display(export_kpi)

✅ Exported clustering_kpi.csv


,n_neighborhoods,n_clusters,silhouette_score,pca_variance_pct,noise_overall_mean,green_overall_mean,study_period,dataset_rows
0,15,3,0.555,95.6,58.72,27.91,"Mar 1–30, 2024",10800


In [27]:
# Summary: list all exported files
import os

print("📁 Export Summary")
print("=" * 60)
for f in sorted(EXPORT_DIR.glob('*.csv')):
    size_kb = os.path.getsize(f) / 1024
    rows = len(pd.read_csv(f))
    print(f"  {f.name:<45} {rows:>3} rows  ({size_kb:.1f} KB)")
print("=" * 60)
print(f"\n📂 All files saved to: {EXPORT_DIR.resolve()}")

📁 Export Summary
  cluster_centers.csv                             3 rows  (0.5 KB)
  clustering_kpi.csv                              1 rows  (0.2 KB)
  elbow_silhouette.csv                            7 rows  (0.2 KB)
  neighborhood_cluster_assignments.csv           15 rows  (1.9 KB)
  neighborhood_silhouette_scores.csv             15 rows  (1.2 KB)
  pca_loadings.csv                                4 rows  (0.3 KB)
  pca_projection.csv                             15 rows  (1.3 KB)

📂 All files saved to: /Users/joaoquintanilha/Downloads/project-hero/reports/kmeans_lab/exports


---

### 📊 How to Build Each Chart in PowerBI

Now that the data is exported, here's a **step-by-step guide** for building each visual in PowerBI Desktop. Follow along in order — each chart builds on the previous imports.

---

#### 0. Import the Data

1. Open **PowerBI Desktop** → **Home** → **Get Data** → **Text/CSV**
2. Import each of the 7 CSV files (one at a time). For each:
   - Navigate to the `exports/` folder
   - Select the file → **Load** (not Transform — the data is already clean)
3. After all imports, go to **Model view** (left sidebar) and verify you see 7 tables
4. No relationships are needed — each table is self-contained

> **💡 Tip:** Rename the tables by right-clicking → **Rename** to remove underscores (e.g., `Neighborhood Cluster Assignments` instead of `neighborhood_cluster_assignments`)

---

#### 1. KPI Cards (Page 1 — Top Banner)

**Visual type:** Card (×4)
**Data source:** `clustering_kpi.csv`

| Step | Action |
|------|--------|
| 1 | Click empty canvas → **Visualizations** pane → **Card** icon |
| 2 | Drag `n_neighborhoods` into the **Fields** well → the card shows "15" |
| 3 | In **Format** pane → **Callout value** → font size 28, color `#1565C0` |
| 4 | **Category label** → turn ON → text = "Neighborhoods" |
| 5 | Resize to a small tile (~200×100 px) and position at top-left |
| 6 | Repeat for `n_clusters` ("Clusters"), `silhouette_score` ("Silhouette Score"), `pca_variance_pct` ("PCA Variance %") |

Arrange the 4 cards in a horizontal row across the top of Page 1.

---

#### 2. Livability Quadrant Scatter (Page 1 — Left Half)

**Visual type:** Scatter Chart
**Data source:** `neighborhood_cluster_assignments.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Scatter Chart** visual |
| 2 | **X Axis** → drag `noise_mean` |
| 3 | **Y Axis** → drag `green_mean` |
| 4 | **Legend** → drag `cluster_name` (this colors points by cluster) |
| 5 | **Details** → drag `nbhd_name` (so each dot = one neighborhood) |
| 6 | **Size** → leave blank or drag `n_obs` for bubble sizing |

**Add quadrant reference lines:**

| Step | Action |
|------|--------|
| 7 | Click the chart → **Format** → **Analytics** (magnifying glass icon) |
| 8 | **X-Axis Constant Line** → **+ Add line** → Value = `58.7`, Color = gray, Style = Dashed, Label = "Mean Noise = 58.7 dB" |
| 9 | **Y-Axis Constant Line** → **+ Add line** → Value = `27.9`, Color = gray, Style = Dashed, Label = "Mean Green = 27.9%" |

**Set cluster colors manually:**

| Step | Action |
|------|--------|
| 10 | **Format** → **Data colors** → click each cluster series |
| 11 | 🟢 Green Suburbs → `#1565C0` |
| 12 | 🔴 Urban Core → `#C62828` |
| 13 | ⚫ Industrial Zone → `#2E7D32` |

**Add data labels:**

| Step | Action |
|------|--------|
| 14 | **Format** → **Data labels** → ON → choose `nbhd_name` as label |
| 15 | Position = Above, font size 8 |

---

#### 3. Cluster Radar / Profile Chart (Page 1 — Right Half)

**Visual type:** Custom visual — **Radar Chart** from AppSource
**Data source:** `cluster_centers.csv`

> **⚠️ Note:** PowerBI doesn't have a built-in radar chart. You need to install one from AppSource.

| Step | Action |
|------|--------|
| 1 | **Home** → **Get More Visuals** (… icon in Visualizations pane) → search "Radar Chart" |
| 2 | Install **"Radar Chart"** by MAQ Software (free) |
| 3 | Insert the Radar Chart visual |
| 4 | **Category** → drag `cluster_name` |
| 5 | **Values** → drag `noise_norm`, `green_norm`, `walk_norm`, `rent_norm` (the 0–1 normalized columns) |
| 6 | **Format** → adjust colors to match: 🟢 `#1565C0`, 🔴 `#C62828`, ⚫ `#2E7D32` |

**Alternative (no AppSource):** Use a **Stacked Bar Chart** with normalized values as a simpler fallback:

| Step | Action |
|------|--------|
| 1 | Unpivot the 4 normalized columns in Power Query: **Transform** → **Unpivot Columns** on `noise_norm`, `green_norm`, `walk_norm`, `rent_norm` |
| 2 | Insert a **Clustered Bar Chart** → Axis = Attribute (feature name), Values = Value, Legend = `cluster_name` |

---

#### 4. Neighborhood Detail Table (Page 1 — Bottom Strip)

**Visual type:** Table (or Matrix)
**Data source:** `neighborhood_cluster_assignments.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Table** visual |
| 2 | Drag these columns in order: `nbhd_name`, `cluster_name`, `noise_mean`, `green_mean`, `walk_mean`, `rent_mean`, `silhouette` |
| 3 | **Format** → **Column headers** → bold, background color `#1565C0`, font color white |
| 4 | **Style** → choose "Alternating rows" for readability |

**Add conditional formatting (color by cluster):**

| Step | Action |
|------|--------|
| 5 | Click the column header `cluster_name` → dropdown arrow → **Conditional formatting** → **Background color** |
| 6 | Format by → **Rules** |
| 7 | Rule 1: If value contains "Green Suburbs" → `#BBDEFB` (light blue) |
| 8 | Rule 2: If value contains "Urban Core" → `#FFCDD2` (light red) |
| 9 | Rule 3: If value contains "Industrial" → `#C8E6C9` (light green) |

---

#### 5. Elbow + Silhouette Dual-Axis Line Chart (Page 2 — Top Left)

**Visual type:** Line Chart (dual Y-axis)
**Data source:** `elbow_silhouette.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Line Chart** visual |
| 2 | **X-Axis** → drag `k` |
| 3 | **Y-Axis** → drag `inertia` |
| 4 | **Secondary Y-Axis** → drag `silhouette` |
| 5 | The chart now shows two lines with independent scales |

**Highlight k=3:**

| Step | Action |
|------|--------|
| 6 | **Format** → **Analytics** → **X-Axis Constant Line** → Value = `3`, Color = `#C62828`, Style = Dotted |
| 7 | Label = "Optimal k=3" |

**Style the lines:**

| Step | Action |
|------|--------|
| 8 | **Format** → **Lines** → Inertia: color `#1565C0`, width 3 |
| 9 | Silhouette: color `#2E7D32`, width 3 |
| 10 | **Markers** → ON for both series (size 6) |

**Add title:** "How Many Clusters? Elbow Method + Silhouette Score"

---

#### 6. Per-Neighborhood Silhouette Bar Chart (Page 2 — Top Right)

**Visual type:** Bar Chart (horizontal)
**Data source:** `neighborhood_silhouette_scores.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Clustered Bar Chart** visual |
| 2 | **Y-Axis** → drag `nbhd_name` |
| 3 | **X-Axis** → drag `silhouette` |
| 4 | **Legend** → drag `cluster_name` |

**Add the average reference line:**

| Step | Action |
|------|--------|
| 5 | **Format** → **Analytics** → **X-Axis Constant Line** → Value = `0.555`, Color = gray, Style = Dashed |
| 6 | Label = "Average = 0.555" |

**Set cluster colors:** Same as chart 2 — `#1565C0`, `#C62828`, `#2E7D32`

**Sort:** Click the chart → **More options** (⋯) → **Sort by** → `silhouette` → **Sort descending**

---

#### 7. PCA Projection Scatter (Page 2 — Bottom Left)

**Visual type:** Scatter Chart
**Data source:** `pca_projection.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Scatter Chart** visual |
| 2 | **X Axis** → drag `pca1` |
| 3 | **Y Axis** → drag `pca2` |
| 4 | **Legend** → drag `cluster_name` |
| 5 | **Details** → drag `nbhd_name` |
| 6 | Set cluster colors: 🟢 `#1565C0`, 🔴 `#C62828`, ⚫ `#2E7D32` |
| 7 | **Data labels** → ON → show `nbhd_name`, position Above |

**Axis labels:** Rename X to "PC1 (74% variance)" and Y to "PC2 (22% variance)" in **Format** → **X Axis** → **Title** → custom text. *(Adjust percentages from your actual PCA output.)*

---

#### 8. Cluster Feature Comparison (Page 2 — Bottom Right)

**Visual type:** Clustered Bar Chart
**Data source:** `cluster_centers.csv`

| Step | Action |
|------|--------|
| 1 | Unpivot the feature columns in Power Query if needed, or use 4 separate bar charts |
| 2 | For a single grouped bar: **Axis** = feature name, **Values** = feature value, **Legend** = `cluster_name` |
| 3 | Alternatively: create 4 small **Card** visuals arranged in a 3×4 grid showing each cluster's value per feature |

---

#### 9. Add the DAX Measures

Go to **Modeling** → **New Measure** and add each measure from the DAX block above:

1. `Cluster Count` — for dynamic KPI if you add slicers later
2. `Avg Silhouette` — for filtered average in tooltips
3. `Noise Above Threshold` — creates an "Above"/"Below" column for conditional formatting
4. `Cluster Color` — use in **conditional formatting** rules for automatic coloring

---

#### 10. Final Polish

| Action | How |
|--------|-----|
| **Page titles** | Text box → "Cluster Overview & Profiles" (Page 1), "Method Validation & Details" (Page 2) |
| **Theme colors** | **View** → **Themes** → **Customize** → Primary = `#1565C0`, Secondary = `#C62828`, Tertiary = `#2E7D32` |
| **Fonts** | Headers: Segoe UI Semibold 14pt, Body: Segoe UI 10pt |
| **Slicer** | Add a **Slicer** with `cluster_name` on both pages for interactive filtering |
| **Background** | **Format** → **Canvas background** → color `#F5F5F5` (light gray) |
| **Navigation** | **Insert** → **Buttons** → **Page navigator** for switching between pages |

> **💡 Teaching Moment:** The difference between a "report" and a "dashboard" in PowerBI: a **report** is what you build in Desktop (multiple pages, interactive). A **dashboard** is a single-page pinned view in the PowerBI Service (cloud). For the lab, we build a report with 2 pages.

---

## Lab Complete ✅

You've completed all steps of the K-Means Clustering Lab:

| Step | Deliverable | Status |
|------|------------|--------|
| **Step 1** | Data Understanding & Clustering Opportunities | ✅ Explored 10,800 rows, identified 4 clustering features |
| **Step 2** | Contextual Visuals | ✅ Scatter plots, correlation heatmaps, radar profiles |
| **Step 3** | Didactic K-Means Implementation | ✅ Elbow, silhouette, PCA, cluster interpretation |
| **Step 4** | PowerBI Data Export | ✅ 7 CSV files exported to `exports/` with full dashboard building guide |

### Key Takeaway

Three clusters emerge from the 15 urban neighborhoods:
- 🔴 **Urban Core** — high noise, high walkability, high rent: the dense downtown trade-off
- 🟢 **Green Suburbs** — quiet, green, moderate cost: the residential ideal
- ⚫ **Industrial Zone** — high noise, low walkability, low rent: areas needing revitalization

This classification gives city planners a data-driven framework for understanding neighborhood character and prioritizing livability interventions.

---
*K-Means Clustering Lab · Urban Neighborhood Livability Analysis*